# Part D: Statistical Uncertainty and Visualization

Product-level bootstrap confidence intervals for five-star rates.

Part D contributor: Zhang Yihui. STAT5106 final project.

## What this part does

Part C reports the five-star rate for each group but gives no uncertainty at all. Part E
reports confidence intervals, but they come from the regression. This part adds the third
thing: intervals estimated without fitting a regression model, under product-cluster sampling assumptions.

All the intervals here come from a bootstrap that resamples products, not reviews. Reviews
of the same product are not independent. They share the ingredients, the price, the brand
and the marketing. Resampling products keeps each product's reviews together, so a product
enters and leaves the sample as a block and the dependence stays in the data.

| Output | File |
| --- | --- |
| Group five-star rates with 95% CIs | `outputs/ci_group_rates.csv` |
| Group differences vs overall | `outputs/ci_differences.csv` |
| Pairwise price comparisons (bootstrap vs naive) | `outputs/ci_pairwise_price.csv` |
| Skin type × price segment cells | `outputs/ci_skin_x_price_cells.csv` |
| Price gradient by skin type + interaction test | `outputs/ci_price_gradient_by_skin.csv`, `outputs/interaction_test.csv` |
| Observed vs model-predicted rates | `outputs/observed_vs_model.csv` |
| Figures | `outputs/figures/` |

Settings: seed 5106, 2000 replicates, percentile intervals. Every table and figure uses the
same 2000 draws, so a difference is computed inside a replicate instead of being put
together from two separate intervals.


In [ ]:
# Resolve this project only; never search the user's home directory.
import sys
from pathlib import Path
_PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                      if (p / "project_paths.py").is_file()), None)
if _PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook within the project folder.")
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))
from project_paths import ROOT, PART_E, RESULTS_DIR, require_analysis, output_dir

import os
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ---- reproducible settings -------------------------------------------------
SEED = 5106
B = 2000                      # bootstrap replicates
LO, HI = 2.5, 97.5            # 95% percentile interval
rng = np.random.default_rng(SEED)

OUT = str(output_dir("D"))
FIG = os.path.join(OUT, "figures")
os.makedirs(FIG, exist_ok=True)

# ---- design system: colours, ink, surfaces ---------------------------------
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#8a8984"
SURFACE, GRID = "#fcfcfb", "#e5e4e0"
BLUE, ORANGE, AQUA, YELLOW, MAGENTA = ("#2a78d6", "#eb6834", "#1baf7a",
                                      "#eda100", "#e87ba4")
BLUE_RAMP = ["#86b6ef", "#5598e7", "#2a78d6", "#184f95"]   # ordinal price ramp
SKIN_COLORS = [BLUE, ORANGE, AQUA, YELLOW, MAGENTA]        # fixed slot order
SKIN_MARKERS = ["o", "s", "^", "D", "v"]                   # secondary encoding

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "savefig.dpi": 200,
    "axes.edgecolor": GRID, "axes.linewidth": 1.0,
    "axes.labelcolor": INK2, "axes.titlecolor": INK,
    "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "font.size": 10, "font.family": "DejaVu Sans",
    "axes.grid": False,
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "legend.fontsize": 9,
    "figure.autolayout": False,
})
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)
print("settings ok · B =", B, "· seed =", SEED)

## 1. Data

Input is Part B's final analysis file (`analysis_ready.parquet`). We only need the
outcome, the three grouping variables and the product identifier used as the cluster.

In [ ]:
DATA_PATH = require_analysis()

df = pd.read_parquet(
    DATA_PATH,
    columns=["product_id", "five_star", "price_segment", "category", "skin_type_cat"],
)

PRICE = ["Budget (<$25)", "Mid ($25-49)", "Premium ($50-99)", "Luxury ($100+)"]
PRICE_SHORT = ["$<25", "$25–49", "$50–99", "$100+"]
SKIN = ["Normal", "Combination", "Dry", "Oily", "Not reported"]
CATEGORY = ["Moisturizers", "Treatments", "Cleansers", "Eye Care", "Masks",
            "Lip Care", "Sunscreen", "Minis & Sets", "Tools, Wellness & Self-Tan"]

# pin the level order so every table, matrix column and figure agrees
df["price_segment"] = pd.Categorical(df["price_segment"], categories=PRICE, ordered=True)
df["skin_type_cat"] = pd.Categorical(df["skin_type_cat"], categories=SKIN, ordered=True)
df["category"] = pd.Categorical(df["category"], categories=CATEGORY, ordered=True)

n = len(df)
P = df["product_id"].nunique()
overall = df["five_star"].mean() * 100

print(f"reviews           {n:,}")
print(f"products (clusters) {P:,}")
print(f"five-star rate    {overall:.2f}%")
print("\nreviews per product:")
sz = df.groupby("product_id", observed=True).size()
print(f"  min {sz.min()}, median {sz.median():.0f}, mean {sz.mean():.1f}, max {sz.max():,}")

# reproducibility record (report rubric: "key programming steps ... to reproduce")
import platform, matplotlib
print(f"\npython {platform.python_version()} · pandas {pd.__version__} · "
      f"numpy {np.__version__} · matplotlib {matplotlib.__version__}")
print(f"input: {os.path.abspath(DATA_PATH)} ({os.path.getsize(DATA_PATH)/1e6:.1f} MB)")
assert n == 1_093_847 and P == 2351
assert df.notna().all().all(), "Missing values or unmapped categorical levels"
assert df.five_star.isin([0, 1]).all()
assert df.groupby("product_id", observed=True)[["price_segment", "category"]].nunique().le(1).all().all()


## 2. Why resample products?

One-way ANOVA diagnostic: between-product sums of squares use the review-weighted grand mean; for binary data, within-product sums of squares equal n × p × (1-p). This diagnostic does not enter the bootstrap.

If reviews were independent, the standard error of a group rate would be
$\sqrt{\hat p(1-\hat p)/n}$, which with $n$ around 100,000 gives very narrow intervals.
That calculation would also be wrong. A product with hundreds of reviews contributes
hundreds of correlated observations, so the real sample size is closer to the number of
products than to the number of reviews.

The intraclass correlation of `five_star` across products measures how strong this is. One
caveat before reading it: the usual rule of thumb $1+(\bar n-1)\text{ICC}$ assumes that
every product has the same number of reviews. Here the counts run from 1 to 16,115, so the
rule of thumb need not equal the actual variance ratio. We compute it anyway as a sanity check and then
measure the design effect directly with the bootstrap in the next section.

In [ ]:
ps = df.groupby("product_id", observed=True)["five_star"].agg(n="size", y="mean")
nbar = ps["n"].mean()
ng = len(ps)
msb = (ps["n"] * (ps["y"] - df["five_star"].mean()) ** 2).sum() / (ng - 1)
msw = (ps["n"] * ps["y"] * (1 - ps["y"])).sum() / (n - ng)
n0 = (n - (ps["n"] ** 2).sum() / n) / (ng - 1)          # unbalanced ANOVA
icc = (msb - msw) / (msb + (n0 - 1) * msw)
de_equal = 1 + (nbar - 1) * icc                         # equal-cluster rule of thumb
size_weighted = (ps["n"] ** 2).sum() / ps["n"].sum()    # size-weighted mean cluster

print(f"ICC (one-way ANOVA, product level)  {icc:.4f}")
print(f"reviews per product: median {ps['n'].median():.0f}, mean {nbar:.0f}, "
      f"size-weighted mean {size_weighted:.0f}, max {ps['n'].max():,}")
print(f"equal-cluster rule of thumb 1+(n̄-1)ICC = {de_equal:.0f}"
      f"  -> naive SEs would be {np.sqrt(de_equal):.1f}x too narrow")
print("Cluster sizes are unequal; use the measured bootstrap variance ratio below.")

## 3. Bootstrap engine

Every replicate draws 2,351 products with replacement and pools all of their reviews.
Building a table of 1.09 million rows 2000 times would be slow, so we first count, for each
product and each group, how many reviews it has ($n_{pg}$) and how many of them are five
star ($y_{pg}$). A replicate is then a weighted sum over the drawn products:

$$\hat p_g^{(b)} = \frac{\sum_k y_{p_k^{(b)} g}}{\sum_k n_{p_k^{(b)} g}}$$

This is the product-level bootstrap, just vectorised. The same draw matrix is reused
everywhere, so every contrast is paired.

In [ ]:
products = np.sort(df["product_id"].unique())
P = len(products)

def count_matrices(frame, key, levels):
    # Product x level matrices of review counts (n) and five-star counts (y)
    g = frame.groupby(["product_id", key], observed=True)["five_star"].agg(n="size", y="sum")
    g = g.unstack(key)
    mat_n = g["n"].reindex(index=products, columns=levels).fillna(0).to_numpy(float)
    mat_y = g["y"].reindex(index=products, columns=levels).fillna(0).to_numpy(float)
    return mat_n, mat_y

# one set of product draws, reused by every table and figure
IDX = rng.integers(0, P, size=(B, P), dtype=np.int32)

def boot_rates(mat_n, mat_y, idx=IDX):
    # B x L matrix of rates; each row is one product-level resample
    out = np.empty((idx.shape[0], mat_n.shape[1]))
    for b in range(idx.shape[0]):
        ii = idx[b]
        denominator = mat_n[ii].sum(0)
        if not np.all(denominator > 0):
            raise ValueError(f"Empty group denominator in bootstrap replicate {b}")
        out[b] = mat_y[ii].sum(0) / denominator
    return out

N_PRICE, Y_PRICE = count_matrices(df, "price_segment", PRICE)
N_SKIN,  Y_SKIN  = count_matrices(df, "skin_type_cat", SKIN)
N_CAT,   Y_CAT   = count_matrices(df, "category", CATEGORY)

boot_price = boot_rates(N_PRICE, Y_PRICE)
boot_skin  = boot_rates(N_SKIN,  Y_SKIN)
boot_cat   = boot_rates(N_CAT,   Y_CAT)

# overall replicate: totals per product, same draws
boot_overall = boot_rates(N_PRICE.sum(1, keepdims=True), Y_PRICE.sum(1, keepdims=True))[:, 0]

print("bootstrap matrix shapes:", boot_price.shape, boot_skin.shape, boot_cat.shape)
print(f"bootstrap mean of overall rate {100*boot_overall.mean():.2f}%  "
      f"(observed {overall:.2f}%)")
print(f"bootstrap SE of overall rate   {100*boot_overall.std(ddof=1):.3f} pp")

## 4. Group five-star rates with 95% intervals

The point estimate comes from the observed data. The interval is the 2.5th and 97.5th
percentile of the bootstrap distribution. The table also gives the bootstrap standard
error, the standard error you would get if reviews were independent, and the ratio between
them.

In [ ]:
from math import erfc, sqrt

Z = 1.959963984540054

def wilson(p, k, z=Z):
    den = 1 + z**2 / k
    centre = (p + z**2 / (2 * k)) / den
    half = z * np.sqrt(p * (1 - p) / k + z**2 / (4 * k**2)) / den
    return centre - half, centre + half

def group_table(group_var, levels, boot, group_name):
    obs = (df.groupby(group_var, observed=True)
             .agg(n_reviews=("five_star", "size"),
                  n_products=("product_id", "nunique"),
                  rate=("five_star", "mean"))
             .reindex(levels))
    p = obs["rate"].to_numpy()
    k = obs["n_reviews"].to_numpy()

    lo, hi = np.percentile(boot, [LO, HI], axis=0)
    se_boot = boot.std(0, ddof=1)
    se_naive = np.sqrt(p * (1 - p) / k)
    wlo, whi = wilson(p, k)

    return pd.DataFrame({
        "group_var": group_name,
        "group": levels,
        "n_reviews": k.astype(int),
        "n_products": obs["n_products"].to_numpy().astype(int),
        "five_star_rate_pct": 100 * p,
        "boot_se_pp": 100 * se_boot,
        "ci_low_pct": 100 * lo,
        "ci_high_pct": 100 * hi,
        "ci_half_width_pp": 100 * (hi - lo) / 2,
        "naive_se_pp": 100 * se_naive,
        "naive_ci_low_pct": 100 * wlo,
        "naive_ci_high_pct": 100 * whi,
        "se_ratio": se_boot / se_naive,
        "design_effect": (se_boot / se_naive) ** 2,
    })

tbl_overall = pd.DataFrame({
    "group_var": ["overall"], "group": ["All reviews"],
    "n_reviews": [n], "n_products": [P],
    "five_star_rate_pct": [overall],
    "boot_se_pp": [100 * boot_overall.std(ddof=1)],
    "ci_low_pct": [100 * np.percentile(boot_overall, LO)],
    "ci_high_pct": [100 * np.percentile(boot_overall, HI)],
    "naive_se_pp": [100 * np.sqrt(overall/100 * (1-overall/100) / n)],
    "se_ratio": [boot_overall.std(ddof=1) / np.sqrt(overall/100*(1-overall/100)/n)],
})
tbl_overall["ci_half_width_pp"] = (tbl_overall["ci_high_pct"] - tbl_overall["ci_low_pct"]) / 2
tbl_overall["design_effect"] = tbl_overall["se_ratio"] ** 2
wlo, whi = wilson(overall/100, n)
tbl_overall["naive_ci_low_pct"] = 100*wlo
tbl_overall["naive_ci_high_pct"] = 100*whi

rates = pd.concat([
    tbl_overall,
    group_table("price_segment", PRICE, boot_price, "price_segment"),
    group_table("skin_type_cat", SKIN, boot_skin, "skin_type"),
    group_table("category", CATEGORY, boot_cat, "category"),
], ignore_index=True)

rates.to_csv(os.path.join(OUT, "ci_group_rates.csv"), index=False)
rates.round(3)

In [ ]:
show = rates[["group_var", "group", "n_reviews", "n_products", "five_star_rate_pct",
              "ci_low_pct", "ci_high_pct", "boot_se_pp", "se_ratio"]].copy()
for c in ["five_star_rate_pct", "ci_low_pct", "ci_high_pct", "boot_se_pp", "se_ratio"]:
    show[c] = show[c].round(2)
r_all = float(rates.loc[0, "se_ratio"])
print(f"Overall five-star rate: {overall:.2f}% "
      f"(95% CI {rates.loc[0,'ci_low_pct']:.2f}–{rates.loc[0,'ci_high_pct']:.2f})")
print(f"Measured design effect for the overall rate: SE ratio {r_all:.1f}x "
      f"(design effect {r_all**2:.0f})")
print(f"Effective sample size ~{n/r_all**2:,.0f} reviews, not {n:,}.")
print(f"Median bootstrap SE / naive SE across all groups: {rates['se_ratio'].median():.1f}x")
display(show)

## 5. Differences from the overall rate

Two overlapping intervals do not tell you that the difference is insignificant, because
the two estimates are correlated. So we bootstrap the difference itself,
$\hat p_g^{(b)} - \hat p_{\text{all}}^{(b)}$, inside each replicate.

In [ ]:
def diff_table(group_var, levels, boot, group_name):
    obs = (df.groupby(group_var, observed=True)["five_star"]
             .agg(n="size", rate="mean").reindex(levels))
    diff = 100 * (obs["rate"].to_numpy() - overall / 100)
    bdiff = 100 * (boot - boot_overall[:, None])
    lo, hi = np.percentile(bdiff, [LO, HI], axis=0)
    p_boot = np.minimum(2 * np.minimum((bdiff <= 0).mean(0), (bdiff >= 0).mean(0)), 1.0)
    return pd.DataFrame({
        "group_var": group_name, "group": levels,
        "n_reviews": obs["n"].to_numpy().astype(int),
        "five_star_rate_pct": 100 * obs["rate"].to_numpy(),
        "diff_vs_overall_pp": diff,
        "diff_ci_low_pp": lo,
        "diff_ci_high_pp": hi,
        "ci_excludes_zero": (lo > 0) | (hi < 0),
        "boot_p": p_boot,
    })

diffs = pd.concat([
    diff_table("price_segment", PRICE, boot_price, "price_segment"),
    diff_table("skin_type_cat", SKIN, boot_skin, "skin_type"),
    diff_table("category", CATEGORY, boot_cat, "category"),
], ignore_index=True)
diffs.to_csv(os.path.join(OUT, "ci_differences.csv"), index=False)

print("Groups whose 95% CI for the difference excludes 0:\n")
print(diffs.loc[diffs["ci_excludes_zero"],
                ["group_var", "group", "diff_vs_overall_pp", "diff_ci_low_pp",
                 "diff_ci_high_pp", "boot_p"]].round(2).to_string(index=False))

## 6. Pairwise price comparisons

In Part C, Budget is 63.88%, Premium 63.89% and Mid 64.27%. These are differences of about
0.4 percentage points. Treated as independent reviews they look significant. Under the
product bootstrap they do not. That difference is the main result of this section.

In [ ]:
def naive_two_prop(p1, n1, p2, n2):
    p = (p1 * n1 + p2 * n2) / (n1 + n2)
    se = sqrt(p * (1 - p) * (1 / n1 + 1 / n2))
    z = (p1 - p2) / se
    return z, erfc(abs(z) / sqrt(2))

obs_price = df.groupby("price_segment", observed=True)["five_star"].agg(n="size", rate="mean")
rows = []
for i in range(len(PRICE)):
    for j in range(i + 1, len(PRICE)):
        pi, ni = obs_price["rate"].iloc[i], obs_price["n"].iloc[i]
        pj, nj = obs_price["rate"].iloc[j], obs_price["n"].iloc[j]
        d = 100 * (boot_price[:, i] - boot_price[:, j])
        lo, hi = np.percentile(d, [LO, HI])
        p_boot = min(2 * min((d <= 0).mean(), (d >= 0).mean()), 1.0)
        z, p_naive = naive_two_prop(pi, ni, pj, nj)
        rows.append({
            "comparison": f"{PRICE[i]} vs {PRICE[j]}",
            "diff_pp": 100 * (pi - pj),
            "boot_ci_low_pp": lo,
            "boot_ci_high_pp": hi,
            "boot_p": p_boot,
            "boot_significant_5pct": (lo > 0) or (hi < 0),
            "naive_z": z,
            "naive_p": p_naive,
            "naive_significant_5pct": p_naive < 0.05,
        })
pairwise = pd.DataFrame(rows)
pairwise.to_csv(os.path.join(OUT, "ci_pairwise_price.csv"), index=False)
pairwise.round(4)

## 7. Skin type × price segment interaction

The question is whether the price gradient is different for different skin types. We
measure the gradient as Luxury minus Budget within each skin type and bootstrap it.

For a test we use the spread of the five gradients, $T = \max_s g_s - \min_s g_s$. If price
really worked differently by skin type, that spread would be large. Under the null the
gradients differ only by sampling noise, so the null distribution of $T$ comes from the
recentred replicates $g_s^{(b)} - \hat g_s$. The test uses the four reported skin types.
"Not reported" is drawn in the figure and excluded from this test by the specified contrast. Part E includes all five skin-type levels and tests 12 adjusted log-odds interactions; these are different null hypotheses.

In [ ]:
df["cell"] = (df["skin_type_cat"].astype(str) + " | " + df["price_segment"].astype(str))
CELLS = [f"{s} | {p}" for s in SKIN for p in PRICE]

N_CELL, Y_CELL = count_matrices(df, "cell", CELLS)
boot_cell = boot_rates(N_CELL, Y_CELL)

obs_cell = (df.groupby("cell", observed=True)["five_star"]
              .agg(n="size", rate="mean").reindex(CELLS))
lo_c, hi_c = np.percentile(boot_cell, [LO, HI], axis=0)

cells_tbl = pd.DataFrame({
    "skin_type": np.repeat(SKIN, 4),
    "price_segment": PRICE * 5,
    "n_reviews": obs_cell["n"].to_numpy().astype(int),
    "five_star_rate_pct": 100 * obs_cell["rate"].to_numpy(),
    "ci_low_pct": 100 * lo_c,
    "ci_high_pct": 100 * hi_c,
    "boot_se_pp": 100 * boot_cell.std(0, ddof=1),
})
cells_tbl["n_products"] = [
    df.loc[df["cell"] == CL, "product_id"].nunique() for CL in CELLS]
cells_tbl.to_csv(os.path.join(OUT, "ci_skin_x_price_cells.csv"), index=False)

# gradient = Luxury (col 3) minus Budget (col 0) within each skin type
G = boot_cell[:, 3::4] - boot_cell[:, 0::4]
g_hat = cells_tbl["five_star_rate_pct"].to_numpy()[3::4] - \
        cells_tbl["five_star_rate_pct"].to_numpy()[0::4]
g_lo, g_hi = 100 * np.percentile(G, [LO, HI], axis=0)

grad = pd.DataFrame({
    "skin_type": SKIN,
    "gradient_luxury_minus_budget_pp": g_hat,
    "ci_low_pp": g_lo,
    "ci_high_pp": g_hi,
    "ci_excludes_zero": (g_lo > 0) | (g_hi < 0),
    "boot_se_pp": 100 * G.std(0, ddof=1),
})
grad.to_csv(os.path.join(OUT, "ci_price_gradient_by_skin.csv"), index=False)

# bootstrap test of no interaction (4 reported skin types)
GS = 100 * G[:, :4]
g_obs = g_hat[:4]
T_obs = g_obs.max() - g_obs.min()
T_boot = (GS - g_obs).max(1) - (GS - g_obs).min(1)
p_int = (1 + (T_boot >= T_obs).sum()) / (B + 1)
inter = pd.DataFrame([{
    "test": "bootstrap range test of equal price gradients across skin types",
    "skin_types": "Normal, Combination, Dry, Oily",
    "statistic_range_pp": T_obs,
    "p_value": p_int,
    "B": B, "seed": SEED,
}])
inter.to_csv(os.path.join(OUT, "interaction_test.csv"), index=False)

print(grad.round(2).to_string(index=False))
print(f"\nInteraction test: T = {T_obs:.2f} pp, bootstrap p = {p_int:.3f}")

## 8. Figure 1: group five-star rates

One panel per grouping variable, all on the same x-axis so the panels can be compared. The
spread of the dots shows how much the grouping matters. The length of the bars shows how
much of that spread could be noise. The vertical line is the overall rate.

In [ ]:
def n_label(nr, npd):
    return f"{nr/1000:.0f}k reviews · {npd} products"

def dot_panel(ax, t, title, xlim, xlabel=False):
    t = t.reset_index(drop=True)
    y = np.arange(len(t))[::-1]
    ax.hlines(y, t["ci_low_pct"], t["ci_high_pct"], color=BLUE, linewidth=2.2,
              alpha=0.40, zorder=2)
    ax.plot(t["five_star_rate_pct"], y, "o", ms=8, color=BLUE,
            mec=SURFACE, mew=1.8, zorder=3)
    ax.axvline(overall, color=MUTED, linewidth=1.1, zorder=1)
    ax.set_yticks(y)
    ax.set_yticklabels([f"{g}\n{n_label(a, b)}" for g, a, b in
                        zip(t["group"], t["n_reviews"], t["n_products"])],
                       fontsize=8.4, linespacing=1.5)
    ax.set_title(title, loc="left", fontsize=10.5, pad=9)
    ax.set_xlim(*xlim)
    if xlabel:
        ax.set_xlabel("Five-star rate (%)", fontsize=9)
    ax.grid(axis="x", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.spines["left"].set_visible(False)
    ax.tick_params(axis="y", length=0)
    ax.tick_params(axis="x", labelsize=9)

fig, axes = plt.subplots(3, 1, figsize=(9.0, 9.6),
                         gridspec_kw={"height_ratios": [3.1, 3.8, 6.9], "hspace": 0.42})
XLIM = (53.5, 71.5)

dot_panel(axes[0], rates[rates.group_var == "price_segment"], "a  Price segment", XLIM)
dot_panel(axes[1], rates[rates.group_var == "skin_type"], "b  Reviewer skin type", XLIM)

cat_t = (rates[rates.group_var == "category"]
         .sort_values("five_star_rate_pct").reset_index(drop=True))
dot_panel(axes[2], cat_t, "c  Product category", XLIM, xlabel=True)

axes[0].annotate(f"All reviews: {overall:.2f}%", xy=(overall, 3.35),
                 xycoords=("data", "data"), fontsize=8.2, color=MUTED, ha="left")

fig.suptitle("Five-star rate by group with 95% product-bootstrap confidence intervals",
             x=0.012, y=0.995, ha="left", fontsize=13, color=INK)
fig.savefig(os.path.join(FIG, "group_rates_ci.png"), bbox_inches="tight")
plt.close(fig)
print("saved group_rates_ci.png")

## 9. Figure 2: interaction plot

Observed five-star rate by price segment, one line per skin type, with a bootstrap interval
at each of the 20 cells. The plot is descriptive. D tests the unadjusted Luxury-minus-Budget risk differences for four reported types. E tests 12 adjusted log-odds interactions across all price and skin levels, so its Wald test addresses a different hypothesis.

In [ ]:
fig, ax = plt.subplots(figsize=(9.4, 6.0))
x = np.arange(4)

for i, s in enumerate(SKIN):
    sub = cells_tbl[cells_tbl.skin_type == s]
    y = sub["five_star_rate_pct"].to_numpy()
    ax.errorbar(x, y, yerr=[y - sub["ci_low_pct"], sub["ci_high_pct"] - y],
                color=SKIN_COLORS[i], marker=SKIN_MARKERS[i], ms=8,
                lw=2.0, elinewidth=1.2, capsize=3, mec=SURFACE, mew=1.6,
                label=s, zorder=3)

ax.set_xticks(x)
ax.set_xticklabels(PRICE_SHORT, fontsize=10)
ax.set_xlabel("Product price segment (USD)", fontsize=10)
ax.set_ylabel("Five-star rate (%)", fontsize=10)
ax.set_xlim(-0.30, 3.30)
ax.set_ylim(min(57, cells_tbl["ci_low_pct"].min() - 0.3), max(71, cells_tbl["ci_high_pct"].max() + 0.3))
ax.grid(axis="y", color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
ax.legend(ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.14),
          handlelength=1.8, columnspacing=1.6, fontsize=9.5)
ax.set_title("Five-star rate by price segment and skin type", loc="left",
             fontsize=12.5, pad=26)
ax.text(0, 1.055, f"95% product-bootstrap intervals · interaction test on the four "
        f"reported skin types: p = {p_int:.2f}",
        transform=ax.transAxes, fontsize=9.5, color=MUTED)
fig.subplots_adjust(top=0.86, bottom=0.24, left=0.08, right=0.98)
fig.savefig(os.path.join(FIG, "interaction_plot.png"), bbox_inches="tight")
plt.close(fig)
print("saved interaction_plot.png")

## 10. Figure 3: the price gradient

It is hard to read "no interaction" off a line chart. This figure plots the gradient
directly: the change in five-star rate from Budget to Luxury, one row per skin type. For
the four reported skin types every interval crosses zero and the points sit within about
1.6 points of each other. "Not reported" is included for completeness and left out of the
test by the specified contrast; E includes it.

In [ ]:
gall = 100 * (boot_price[:, 3] - boot_price[:, 0])
gall_obs = 100 * (obs_price["rate"].iloc[3] - obs_price["rate"].iloc[0])

fig, ax = plt.subplots(figsize=(8.8, 4.9))
ys = np.arange(len(SKIN))[::-1].astype(float)
for yy, (_, r) in zip(ys, grad.iterrows()):
    ax.hlines(yy, r["ci_low_pp"], r["ci_high_pp"], color=BLUE, linewidth=2.4,
              alpha=0.40, zorder=2)
    ax.hlines(yy, 0, r["gradient_luxury_minus_budget_pp"], color=GRID, linewidth=1.0, zorder=1)
    ax.plot(r["gradient_luxury_minus_budget_pp"], yy, "o", ms=9, color=BLUE,
            mec=SURFACE, mew=1.8, zorder=3)

y_all = -1.45
ax.hlines(y_all, *np.percentile(gall, [LO, HI]), color=MUTED, linewidth=2.4,
          alpha=0.60, zorder=2)
ax.plot(gall_obs, y_all, "o", ms=9, color=INK2, mec=SURFACE, mew=1.8, zorder=3)

ax.axvline(0, color=INK2, linewidth=1.0, zorder=1)
ax.set_yticks(list(ys) + [y_all])
ax.set_yticklabels(list(grad["skin_type"]) + ["All reviewers"], fontsize=9.5)
ax.set_xlabel("Change in five-star rate, Luxury − Budget (percentage points)", fontsize=9.5)
ax.set_xlim(-10.5, 3.4)
ax.set_ylim(-2.1, 4.6)
ax.grid(axis="x", color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
ax.spines["left"].set_visible(False)
ax.tick_params(axis="y", length=0)
ax.text(3.2, 0, "excluded from\ninteraction test", fontsize=8, color=MUTED,
        ha="right", va="center")
ax.set_title("Observed Luxury − Budget differences by skin type", loc="left",
             fontsize=12.5, pad=26)
ax.text(0, 1.035, "Luxury minus Budget, percentage points · 95% product-bootstrap "
        "intervals", transform=ax.transAxes, fontsize=9.3, color=MUTED)
fig.savefig(os.path.join(FIG, "price_gradient_by_skin.png"), bbox_inches="tight")
plt.close(fig)
print("saved price_gradient_by_skin.png")

## 11. Figure 4: naive vs product-level intervals

The thin grey bars are Wilson intervals, which assume the reviews are independent. The blue
bars are the product bootstrap. The point estimates are the same; the uncertainty is not.
The naive intervals are 3 to 11 times too narrow (median 8), which is why most of the
one-to-three-point differences in Part C cannot be separated from noise.

In [ ]:
fig, ax = plt.subplots(figsize=(9.8, 5.4))
t = rates[rates.group_var.isin(["overall", "price_segment", "skin_type"])].copy()
t["lab"] = np.where(t.group_var == "overall", "All reviews",
                    np.where(t.group_var == "price_segment", "Price · " + t["group"],
                             "Skin · " + t["group"]))
t = t.reset_index(drop=True)
y = np.arange(len(t))[::-1].astype(float)
OFF = 0.20

ax.hlines(y + OFF, t["naive_ci_low_pct"], t["naive_ci_high_pct"], color=MUTED,
          linewidth=2.2, zorder=2)
ax.hlines(y - OFF, t["ci_low_pct"], t["ci_high_pct"], color=BLUE,
          linewidth=3.0, zorder=3)
ax.plot(t["five_star_rate_pct"], y, "o", ms=6.5, color=INK, mec=SURFACE,
        mew=1.2, zorder=4)
ax.axvline(overall, color=GRID, linewidth=1.2, zorder=1)

ax.set_yticks(y)
ax.set_yticklabels(t["lab"], fontsize=9.2)
ax.set_xlabel("Five-star rate (%)", fontsize=9.5)
ax.set_xlim(55.0, 70.0)
ax.set_ylim(-0.8, len(t) - 0.2)
ax.grid(axis="x", color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
ax.spines["left"].set_visible(False)
ax.tick_params(axis="y", length=0)

for yy, r in zip(y, t["se_ratio"]):
    ax.text(1.015, yy, f"×{r:.1f}", transform=ax.get_yaxis_transform(),
            fontsize=8.6, color=INK2, va="center", ha="left", clip_on=False)
ax.text(1.015, y[0] + 0.62, "SE ratio", transform=ax.get_yaxis_transform(),
        fontsize=8.2, color=MUTED, va="center", ha="left", clip_on=False)

from matplotlib.lines import Line2D
ax.legend(handles=[
    Line2D([0], [0], color=MUTED, lw=2.2,
           label="Naive Wilson 95% CI — reviews treated as independent"),
    Line2D([0], [0], color=BLUE, lw=3.0,
           label="Product-bootstrap 95% CI — reviews resampled by product"),
], loc="upper center", bbox_to_anchor=(0.5, -0.17), ncol=2, fontsize=9)
ax.set_title("The same rates under two uncertainty assumptions", loc="left",
             fontsize=12.5, pad=26)
ax.text(0, 1.045, "Intervals offset vertically for visibility · SE ratio = bootstrap SE ÷ "
        "naive SE", transform=ax.transAxes, fontsize=9.3, color=MUTED)
fig.savefig(os.path.join(FIG, "naive_vs_cluster_ci.png"), bbox_inches="tight")
plt.close(fig)
print("saved naive_vs_cluster_ci.png")

## 12. Figure 5: observed rates vs Part E

The figure compares raw cell rates with E's standardized adjusted predictions. They use different covariate mixtures and the same reviews. This is a descriptive comparison, not an equivalence test or independent model validation.


In [ ]:
MODEL_CSV_CANDIDATES = [output_dir("E") / "predicted_probabilities_skin_x_price.csv"]
MODEL_PATH = next((p for p in MODEL_CSV_CANDIDATES if p.is_file()), None)
if MODEL_PATH is None:
    raise FileNotFoundError("Part E prediction CSV is required for the D/E consistency check.")

if MODEL_PATH is None:
    print("Part E predicted probabilities not found — skipping Figure 5.")
else:
    model = pd.read_csv(MODEL_PATH)
    assert len(model) == 20 and not model.duplicated(["skin", "price"]).any()
    assert model[["pred", "low", "high"]].notna().all().all()
    assert (model["low"] <= model["pred"]).all() and (model["pred"] <= model["high"]).all()
    model["price_segment"] = model["price"].map(
        {"Budget": PRICE[0], "Mid": PRICE[1], "Premium": PRICE[2], "Luxury": PRICE[3]})

    cmp_rows = []
    for s in SKIN:
        for p in PRICE:
            o = cells_tbl[(cells_tbl.skin_type == s) & (cells_tbl.price_segment == p)].iloc[0]
            m = model[(model.skin == s) & (model.price_segment == p)].iloc[0]
            cmp_rows.append({"skin_type": s, "price_segment": p,
                             "observed_pct": o["five_star_rate_pct"],
                             "observed_ci_low_pct": o["ci_low_pct"],
                             "observed_ci_high_pct": o["ci_high_pct"],
                             "model_pred_pct": 100*m["pred"],
                             "model_ci_low_pct": 100*m["low"],
                             "model_ci_high_pct": 100*m["high"]})
    cmp_tbl = pd.DataFrame(cmp_rows)
    cmp_tbl.to_csv(os.path.join(OUT, "observed_vs_model.csv"), index=False)

    fig, axes = plt.subplots(2, 3, figsize=(12.6, 6.4), sharey=True, sharex=True)
    xs = np.arange(4)
    for i, s in enumerate(SKIN):
        ax = axes.flat[i]
        sub = cmp_tbl[cmp_tbl.skin_type == s]
        ax.fill_between(xs, sub["model_ci_low_pct"], sub["model_ci_high_pct"],
                        color=ORANGE, alpha=0.13, linewidth=0, zorder=1)
        ax.plot(xs, sub["model_pred_pct"], "-", color=ORANGE, lw=2, zorder=2,
                label="Model prediction")
        ax.errorbar(xs, sub["observed_pct"],
                    yerr=[sub["observed_pct"]-sub["observed_ci_low_pct"],
                          sub["observed_ci_high_pct"]-sub["observed_pct"]],
                    fmt="o", color=BLUE, ms=7, capsize=3, elinewidth=1.2,
                    mec=SURFACE, mew=1.4, zorder=3, label="Observed (bootstrap CI)")
        ax.set_title(s, loc="left", fontsize=10.5)
        ax.set_xticks(xs)
        ax.set_xticklabels(PRICE_SHORT, fontsize=8.5)
        ax.set_ylim(min(56, cmp_tbl["observed_ci_low_pct"].min()-0.3, cmp_tbl["model_ci_low_pct"].min()-0.3),
                    max(71, cmp_tbl["observed_ci_high_pct"].max()+0.3, cmp_tbl["model_ci_high_pct"].max()+0.3))
        ax.grid(axis="y", color=GRID, linewidth=0.8)
        ax.set_axisbelow(True)
    ax_empty = axes.flat[5]
    ax_empty.axis("off")
    h, l = axes.flat[0].get_legend_handles_labels()
    ax_empty.legend(h, l, loc="center left", fontsize=10, handlelength=2.2,
                    labelspacing=1.1)
    for ax in axes[:, 0]:
        ax.set_ylabel("Five-star rate (%)", fontsize=9)
    for row_i in range(axes.shape[0]):
        for col_i in range(axes.shape[1]):
            axes[row_i, col_i].tick_params(labelleft=(col_i == 0))

    fig.suptitle("Observed cell rates and adjusted Part E predictions",
                 x=0.008, y=1.03, ha="left", fontsize=13)
    fig.tight_layout()
    fig.savefig(os.path.join(FIG, "observed_vs_model.png"), bbox_inches="tight")
    plt.close(fig)
    print("saved observed_vs_model.png")

## 13. Key findings

All of this is printed from the tables above.

In [ ]:
tmp = rates.set_index("group_var")
sig = diffs[diffs["ci_excludes_zero"]].copy()

print("1. Uncertainty is much larger than the naive calculation suggests")
print(f"   ICC at product level {icc:.3f}. Measured design effect for the overall rate:")
print(f"   SE ratio {r_all:.1f}x, so the effective sample size is about "
      f"{n/r_all**2:,.0f} reviews, not {n:,}.")
print(f"   Overall five-star rate {overall:.2f}% "
      f"(95% CI {tmp.loc['overall','ci_low_pct']:.2f}–{tmp.loc['overall','ci_high_pct']:.2f}).")
print()
print("2. Price: Luxury sits about 3pp below the rest, but the interval is wide")
print("   (Luxury vs overall excludes 0 only just; see the pairwise table in section 6)")
for _, r in rates[rates.group_var == "price_segment"].iterrows():
    print(f"   {r['group']:<18s} {r['five_star_rate_pct']:5.2f}%  "
          f"95% CI {r['ci_low_pct']:5.2f}–{r['ci_high_pct']:5.2f}")
print()
print("3. Skin type: the four reported types span only 1.2pp and overlap heavily")
for _, r in rates[rates.group_var == "skin_type"].iterrows():
    print(f"   {r['group']:<14s} {r['five_star_rate_pct']:5.2f}%  "
          f"95% CI {r['ci_low_pct']:5.2f}–{r['ci_high_pct']:5.2f}")
print()
print("4. Category: the largest and most robust differences")
for _, r in (rates[rates.group_var == "category"]
             .sort_values("five_star_rate_pct")).iterrows():
    star = " *" if r["group"] in set(sig.loc[sig.group_var=="category","group"]) else ""
    print(f"   {r['group']:<26s} {r['five_star_rate_pct']:5.2f}%  "
          f"95% CI {r['ci_low_pct']:5.2f}–{r['ci_high_pct']:5.2f}{star}")
print()
print("5. Pairwise price comparisons: bootstrap vs naive")
for _, r in pairwise.iterrows():
    print(f"   {r['comparison']:<40s} {r['diff_pp']:+.2f}pp  "
          f"boot p={r['boot_p']:.3f} ({'sig' if r['boot_significant_5pct'] else 'ns'})"
          f"   naive p={r['naive_p']:.2e} ({'sig' if r['naive_significant_5pct'] else 'ns'})")
print(f"   Naive SEs call {pairwise['naive_significant_5pct'].sum()} of 6 pairs "
      f"'significant'; clustered intervals call "
      f"{pairwise['boot_significant_5pct'].sum()}.")
print()
print("6. Interaction: no evidence the price gradient varies by skin type")
for _, r in grad.iterrows():
    print(f"   {r['skin_type']:<14s} {r['gradient_luxury_minus_budget_pp']:+5.2f}pp  "
          f"95% CI {r['ci_low_pp']:+.2f} to {r['ci_high_pp']:+.2f}")
print(f"   Bootstrap range test: T = {T_obs:.2f} pp, p = {p_int:.3f} "
      f"(Part E Wald test p = 0.50 — same conclusion from two different routes)")

## 14. Which rate are we estimating?

The bootstrap resamples products, but the quantity it estimates still weights each review
equally, so a product with 10,000 reviews counts more than one with 10. That is the same raw-rate target as Part C. Part E estimates adjusted odds and standardized probabilities from review-level models. A rate where every product counts once answers a different
question and gives different numbers, so we report it only as a check.

In [ ]:
pw = (df.groupby(["product_id", "price_segment"], observed=True)["five_star"].mean()
        .groupby(level="price_segment", observed=True).mean().reindex(PRICE))
rw = df.groupby("price_segment", observed=True)["five_star"].mean().reindex(PRICE)
pw_all = df.groupby("product_id", observed=True)["five_star"].mean().mean()
print("                           review-weighted   product-weighted")
print(f"All reviews                {overall:14.2f}%  {100*pw_all:15.2f}%")
for p_ in PRICE:
    print(f"{p_:<26s} {100*rw[p_]:14.2f}%  {100*pw[p_]:15.2f}%")

print("\nThe weighting changes the estimand and the price ordering: Luxury is lowest")
print("(%.1f%%), but second-highest by products (%.1f%%), below Premium."
      % (100*rw[PRICE[3]], 100*pw[PRICE[3]]))
print("Review-count weighting changes the relative contribution of products.")
print("The Luxury gap is a statement about reviews in this sample, not")
print("about products in general. Put this weighting sensitivity in the Limitations.")
# Compact bootstrap settings and variance diagnostics.
import hashlib
weighting = pd.DataFrame({"price_segment": PRICE, "review_weighted_pct": 100*rw.to_numpy(),
                          "product_weighted_pct": 100*pw.to_numpy()})
weighting.to_csv(os.path.join(OUT, "weighting_sensitivity.csv"), index=False)
diagnostics = {
    "seed": SEED, "replicates": B, "products": P, "reviews": n,
    "percentile_method": "linear", "icc": float(icc), "ms_between": float(msb), "ms_within": float(msw),
    "review_weighted_overall_pct": float(overall), "product_weighted_overall_pct": float(100*pw_all),
    "overall_bootstrap_se_pp": float(100*boot_overall.std(ddof=1)),
    "overall_design_effect": float(r_all**2), "effective_review_count_variance_equivalent": float(n/r_all**2),
    "draw_indices_sha256": hashlib.sha256(IDX.tobytes()).hexdigest(),
    "interaction_p": float(p_int), "interaction_monte_carlo_se": float(np.sqrt(p_int*(1-p_int)/B)),
    "part_e_predictions_path": str(Path(MODEL_PATH).relative_to(ROOT)) if Path(MODEL_PATH).is_relative_to(ROOT) else str(MODEL_PATH),
}
with open(os.path.join(OUT, "bootstrap_diagnostics.json"), "w") as stream:
    json.dump(diagnostics, stream, indent=2)
    stream.write("\n")
